In [1]:
# Define the list of “politics” topic IDs you care about and other arguments
politics_topics = [
    0, 3, 5, 6, 7, 8, 10, 12, 13, 18, 20, 22, 23, 26, 28, 30, 33, 37, 38, 39,
    40, 43, 44, 47, 49, 51, 53, 58, 62, 63, 64, 65, 68, 72, 74, 78, 79, 85, 87,
    89, 90, 92, 96, 100, 104, 106, 107, 108, 110, 111, 114, 116, 118, 119, 121,
    122, 124, 125, 126, 128, 131, 132
]

economy_topics = [2, 11, 24, 25, 31, 54, 60, 66, 73, 101, 113, 130]

crypto_topics = [1, 4, 14, 21, 32, 35, 55, 56, 59, 67, 80, 84, 86]

not_categorized_topics = [
    9, 15, 16, 17, 19, 27, 29, 34, 36, 41, 42, 45, 46, 48, 50, 52, 57, 61, 69,
    70, 71, 75, 76, 77, 81, 82, 83, 88, 91, 93, 94, 95, 97, 98, 99, 102, 103,
    105, 109, 112, 115, 117, 120, 123, 127, 129
]



In [2]:
level_depth = 0

In [3]:
import os
import argparse
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
import plotly.io as pio
from pathlib import Path
from bertopic import BERTopic
import pandas as pd
import plotly.io as pio
from bertopic import BERTopic
from pathlib import Path
import pandas as pd
import plotly.express as px
import pandas as pd
import numpy as np

# Headless-safe Plotly renderer
pio.renderers.default = "notebook_connected" if (os.environ.get("DISPLAY") or os.environ.get("MPLBACKEND")) else "png"

print(f"[INFO] level_depth={level_depth}")

/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] level_depth=0


In [4]:
# DF_SAMPLED
# here we are working only with the SEEDS

#paths
path_df_sampled = f"../results/levels/level_{level_depth}/grid_search/df_sampled_level_{level_depth}.csv"
best_dir = f"../results/levels/level_{level_depth}/best"
best_model_path = os.path.join(best_dir, "best_model.pkl")
df_spam_messages_path = f"../results/levels/level_{level_depth}/preProcessing/preprocessed_spam_messages_level_{level_depth}.tsv.gz"
path_summary_filtered = f"../results/levels/level_{level_depth}/percentage_of_politics_msgs/summary_pol_sorted_filtered_spam.csv.gz"


# Load model and dataframe
topic_model = BERTopic.load(best_model_path)
df_sampled = pd.read_csv(path_df_sampled)


# Attach the assigned topic to each message
df_sampled["topic"] = topic_model.topics_
print("len(df_sampled) =", len(df_sampled))

#DF_SPAM_MESSAGES
# check variable considered_spam_threshold in script 1_preProcessing,oy
df_spam_messages = pd.read_csv(df_spam_messages_path, sep='\t', compression='gzip')

"""
df_spam_messages
| channel\_id | spam_msg                  | count_spam |
| ----------- | ------------------------- | -----      |
| 1001        | join our crypto group now | 2          |
| 1001        | hello everyone            | 1          |
| 1002        | vote for freedom          | 3          |
| 1002        | hello                     | 1          |
"""


#DF_TOTAL_SPAM
df_spam_messages["count_spam"] = df_spam_messages["count_spam"].clip(lower=0)  
df_total_spam = (
    df_spam_messages
    .groupby("channel_id")["count_spam"]
    .sum()
    .reset_index(name="spam_msgs")
)



'''
| channel_id | spam_msgs |
|------------|-----------|
| 1001       | 3         |
| 1002       | 4         |

'''
print(f"len df_spam=\n{len(df_spam_messages)}")
print(f"len df_total_spam=\n{len(df_total_spam)}")
print(f"df_spam=\n{df_spam_messages.head(10)}")
print(f"df_total_spam=\n{df_total_spam.head(10)}")

len(df_sampled) = 30054
len df_spam=
7695
len df_total_spam=
23
df_spam=
           channel_id               text_preprocessed  count_spam
0  channel_1076871110                         exactly          66
1  channel_1076871110                           trump          49
2  channel_1076871110                       president          47
3  channel_1076871110                          course          46
4  channel_1076871110     thread reader thread reader          45
5  channel_1076871110                            mean          43
6  channel_1076871110                           right          42
7  channel_1076871110                           india          40
8  channel_1076871110                            know          37
9  channel_1076871110  country fault business problem          34
df_total_spam=
           channel_id  spam_msgs
0  channel_1076871110      28273
1  channel_1245638927        520
2  channel_1308756973         14
3  channel_1410320603         10
4  channel_1425465243

In [5]:

# Group by channel_id and compute:
#    - total_msgs_without_spam: total number of messages in that channel
#    - pol_msgs:   how many of those messages belong to the politics_topics
#    - merge with spam
# grouping

#removing spam messages from df_sampled
spam_texts = set(df_spam_messages["text_preprocessed"])
df_sampled_without_spam = df_sampled[~df_sampled["text_preprocessed"].isin(spam_texts)].copy()
summary = (
    df_sampled_without_spam
    .groupby("channel_id")["topic"]
    .agg(
        total_msgs_without_spam   = "count",
        pol_msgs     = lambda s: s.isin(politics_topics).sum(),
        economy_msgs = lambda s: s.isin(economy_topics).sum(),
        crypto_msgs  = lambda s: s.isin(crypto_topics).sum(),
        not_categorized_msgs   = lambda s: s.isin(not_categorized_topics).sum(),
    )
    .reset_index()
)

#merging
summary = summary.merge(df_total_spam, on="channel_id", how="left" )
summary["spam_msgs"] = summary["spam_msgs"].fillna(0).astype(int)

#adding spam count 
summary["outliers_msgs"] = summary["total_msgs_without_spam"] - summary["pol_msgs"] - summary["economy_msgs"] - summary["crypto_msgs"] - summary["not_categorized_msgs"] 
summary["total_msgs_with_spam"] = summary["total_msgs_without_spam"] + summary["spam_msgs"] 

# calculate all the ratio
summary = summary.assign(
    pol_ratio_without_spam     = lambda d: d["pol_msgs"]   / d["total_msgs_without_spam"] * 100,
    economy_ratio_without_spam = lambda d: d["economy_msgs"]/ d["total_msgs_without_spam"] * 100,
    crypto_ratio_without_spam  = lambda d: d["crypto_msgs"]/ d["total_msgs_without_spam"] * 100,
    not_categorized_ratio_without_spam   = lambda d: d["not_categorized_msgs"] / d["total_msgs_without_spam"] * 100,
    spam_ratio    = lambda d: d["spam_msgs"] / d["total_msgs_with_spam"] * 100
)

# Filter: only channels with more than 100 total messages
summary_pol_sorted_filtered_spam = summary[summary["total_msgs_without_spam"] > 10]

# Sort by political message ratio
summary_pol_sorted = summary_pol_sorted_filtered_spam.sort_values(by="pol_ratio_without_spam", ascending=False)

# Define the bins and the truncated % labels
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = ["0%", "10%", "20%", "30%", "40%", "50%", "60%", "70%", "80%", "90%"]

# Assign each channel to a bin based on political ratio
summary_pol_sorted_filtered_spam["pol_ratio_without_spam_range"] = pd.cut(summary_pol_sorted_filtered_spam["pol_ratio_without_spam"], bins=bins, labels=labels, right=True, include_lowest=True)

#save summary filtered  
Path(path_summary_filtered).parent.mkdir(parents=True, exist_ok=True)
summary_pol_sorted_filtered_spam.to_csv(path_summary_filtered, sep=",", index=False, compression="gzip")
print(f"len summary_pol_sorted_filtered_spam:\n{len(summary_pol_sorted_filtered_spam)}")
print(f"summary_pol_sorted_filtered_spam:\n{summary_pol_sorted_filtered_spam}")

'''
summary_pol_sorted_filtered_spam=
   channel_id  total_msgs_without_spam  pol_msgs  economy_msgs  crypto_msgs  \
0  1840578235                      540       310            45           12   
1  2036850729                      260       120            12            8   
2  1306559115                     1240       780            40           15   
3  1749991917                      180        55            18            9   
4  1581117699                      415        92            33           60   

   not_categorized_msgs  spam_msgs  outliers_msgs  total_msgs_with_spam  \
0                   150         60             23                   600   
1                   110         40             10                   300   
2                   360        155             45                  1395   
3                    85         20             13                   200   
4                   205         35             25                   450   

   pol_ratio_without_spam  economy_ratio_without_spam  \
0                   57.41                        8.33   
1                   46.15                        4.62   
2                   62.90                        3.23   
3                   30.56                       10.00   
4                   22.17                        7.95   

   crypto_ratio_without_spam  not_categorized_ratio_without_spam  spam_ratio  \
0                       2.22                                27.78      10.00   
1                       3.08                                42.31      13.33   
2                       1.21                                29.03      11.11   
3                       5.00                                47.22      10.00   
4                      14.46                                49.40       7.78   

  pol_ratio_without_spam_range
0                         50%
1                         40%
2                         60%
3                         30%
4                         20%
'''

len summary_pol_sorted_filtered_spam:
43
summary_pol_sorted_filtered_spam:
            channel_id  total_msgs_without_spam  pol_msgs  economy_msgs  \
0   channel_1076871110                      555       211            20   
2   channel_1245638927                      914       403            26   
3   channel_1269157403                      142        36             3   
4   channel_1283801046                       19         7             0   
5   channel_1292024994                       78        41             2   
6   channel_1308756973                      760       237            20   
7   channel_1314402837                       11         0             0   
9   channel_1355219851                       73        43             2   
10  channel_1382481365                     2945      1953            55   
11  channel_1410320603                      608       274            24   
12  channel_1425465243                     2076      1201           380   
13  channel_1431327903   

'\nsummary_pol_sorted_filtered_spam=\n   channel_id  total_msgs_without_spam  pol_msgs  economy_msgs  crypto_msgs  0  1840578235                      540       310            45           12   \n1  2036850729                      260       120            12            8   \n2  1306559115                     1240       780            40           15   \n3  1749991917                      180        55            18            9   \n4  1581117699                      415        92            33           60   \n\n   not_categorized_msgs  spam_msgs  outliers_msgs  total_msgs_with_spam  0                   150         60             23                   600   \n1                   110         40             10                   300   \n2                   360        155             45                  1395   \n3                    85         20             13                   200   \n4                   205         35             25                   450   \n\n   pol_ratio_without_spam  e